In [5]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [7]:
# Define the quantum states for 0 and 1 in the computational basis (Z-basis)
# |0> state is the default, so an empty circuit represents it.
qc_0_z = QuantumCircuit(1, name='|0>_Z')
zero_state_z = circuit_to_gate(qc_0_z)

# |1> state: apply an X gate to the |0> state
qc_1_z = QuantumCircuit(1, name='|1>_Z')
qc_1_z.x(0)
one_state_z = circuit_to_gate(qc_1_z)

# Define the quantum states for 0 and 1 in the Hadamard basis (X-basis)
# |+> state (which is |0> in X-basis)
qc_0_x = QuantumCircuit(1, name='|+>_X')
qc_0_x.h(0)
zero_state_x = circuit_to_gate(qc_0_x)

# |-> state (which is |1> in X-basis)
qc_1_x = QuantumCircuit(1, name='|->_X')
qc_1_x.x(0)
qc_1_x.h(0)
one_state_x = circuit_to_gate(qc_1_x)

print("Defined quantum states for 0 and 1 in Z and X bases as gates.")

Defined quantum states for 0 and 1 in Z and X bases as gates.


In [8]:
# Define measurement circuits for Z-basis (computational basis)
# Measurement circuits cannot be directly converted to gates using circuit_to_gate
# because they involve classical bits. We will use the QuantumCircuit objects directly.
measure_z = QuantumCircuit(1, 1, name='Measure_Z')
measure_z.measure(0, 0)

# Define measurement circuits for X-basis (Hadamard basis)
# To measure in X-basis, apply Hadamard gate, then measure in Z-basis
measure_x = QuantumCircuit(1, 1, name='Measure_X')
measure_x.h(0)
measure_x.measure(0, 0)

print("Defined measurement circuits for Z and X bases.")

Defined measurement circuits for Z and X bases.


In [9]:
import random

def alice_prepares_qubit(bit, basis):
    """
    Alice prepares a qubit based on her bit and chosen basis.
    bit: 0 or 1
    basis: 'Z' or 'X'
    Returns a QuantumCircuit representing the prepared qubit.
    """
    qc = QuantumCircuit(1, name=f"Alice_State_{bit}_{basis}")
    if basis == 'Z':
        if bit == 0:
            # |0> state is default, so nothing to do
            pass
        else:
            # |1> state
            qc.x(0)
    elif basis == 'X':
        if bit == 0:
            # |+> state
            qc.h(0)
        else:
            # |-> state
            qc.x(0)
            qc.h(0)
    else:
        raise ValueError("Invalid basis for Alice")
    return qc

def bob_measures_qubit(qubit_circuit, basis):
    """
    Bob measures a received qubit in his chosen basis.
    qubit_circuit: QuantumCircuit object representing the received qubit.
    basis: 'Z' or 'X'
    Returns the measurement outcome (0 or 1) and the measured circuit.
    """
    num_qubits = qubit_circuit.num_qubits
    num_clbits = 1 # Bob always measures into one classical bit

    # Create a new circuit for measurement, preserving Alice's state prep
    measurement_circuit = QuantumCircuit(num_qubits, num_clbits)

    # Append Alice's state preparation to the measurement circuit
    measurement_circuit.append(qubit_circuit.to_instruction(), [0])

    if basis == 'Z':
        measurement_circuit.measure(0, 0)
    elif basis == 'X':
        measurement_circuit.h(0) # Apply Hadamard before measuring in Z-basis for X-basis measurement
        measurement_circuit.measure(0, 0)
    else:
        raise ValueError("Invalid basis for Bob")

    simulator = BasicSimulator()
    compiled_circuit = transpile(measurement_circuit, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(compiled_circuit)

    # Get the measurement outcome (the key will be '0' or '1' string)
    outcome = int(list(counts.keys())[0])

    return outcome, measurement_circuit

print("Alice's qubit preparation and Bob's measurement functions defined.")

Alice's qubit preparation and Bob's measurement functions defined.


In [10]:
# Number of bits to generate
num_bits = 100

# Alice's data
alice_bits = [random.randint(0, 1) for _ in range(num_bits)]
alice_bases = [random.choice(['Z', 'X']) for _ in range(num_bits)]

# Bob's data
bob_bases = [random.choice(['Z', 'X']) for _ in range(num_bits)]
bob_measured_bits = []

# Store the circuits for visualization (optional)
quantum_circuits_alice = []
quantum_circuits_bob_measured = []

for i in range(num_bits):
    # Alice prepares qubit
    alice_qubit_circuit = alice_prepares_qubit(alice_bits[i], alice_bases[i])
    quantum_circuits_alice.append(alice_qubit_circuit)

    # Bob measures qubit
    bob_outcome, measured_circuit = bob_measures_qubit(alice_qubit_circuit, bob_bases[i])
    bob_measured_bits.append(bob_outcome)
    quantum_circuits_bob_measured.append(measured_circuit)

# --- Sifting the Key --- #
alice_sifted_key = []
bob_sifted_key = []
matching_bases_indices = []

for i in range(num_bits):
    if alice_bases[i] == bob_bases[i]:
        matching_bases_indices.append(i)
        alice_sifted_key.append(alice_bits[i])
        bob_sifted_key.append(bob_measured_bits[i])

print(f"Alice's original bits: {alice_bits}")
print(f"Alice's chosen bases:  {alice_bases}")
print(f"Bob's chosen bases:    {bob_bases}")
print(f"Bob's measured bits:   {bob_measured_bits}")
print("\n--- Sifted Key ---")
print(f"Alice's sifted key: {alice_sifted_key}")
print(f"Bob's sifted key:   {bob_sifted_key}")

# Verify if the sifted keys match (ideally, they should if no eavesdropper)
num_matches = sum(1 for a, b in zip(alice_sifted_key, bob_sifted_key) if a == b)
key_length = len(alice_sifted_key)
match_percentage = (num_matches / key_length * 100) if key_length > 0 else 0

print(f"\nKey Length: {key_length}")
print(f"Number of matching bits: {num_matches}")
print(f"Match percentage: {match_percentage:.2f}%")

if match_percentage == 100:
    print("Alice and Bob's sifted keys match perfectly!")
else:
    print("There are discrepancies in the sifted keys. This could indicate an eavesdropper or measurement errors.")

Alice's original bits: [0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
Alice's chosen bases:  ['X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'Z', 'X', 'X', 'X', 'X', 'Z', 'X', 'X', 'X', 'X', 'X', 'X', 'Z', 'X', 'X', 'Z', 'X', 'X', 'X', 'X', 'Z', 'Z', 'Z', 'X', 'X', 'X', 'Z', 'Z', 'X', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'X', 'X', 'Z', 'X', 'X', 'X', 'Z', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'Z', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'X', 'Z', 'Z', 'X', 'X', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'X', 'X', 'X', 'X', 'X', 'Z', 'Z', 'X', 'Z']
Bob's chosen bases:    ['X', 'X', 'X', 'Z', 'Z', 'Z', 'Z', 'X', 'X', 'X', 'X', 'X', 'Z', 'Z', 'Z', 'X', 'X', 'X', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'Z', 'Z'

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

